In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

import os

In [ ]:
base_model_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_v4"
new_base_model_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_new_method/"
fold = 3
testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
input_file_path = os.path.join(testing_dir, 'cnn_positive_patch_3points.txt')
testing_dir_2 = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
output_file_path = os.path.join(testing_dir_2, 'testing_patch_index.csv')
df_dir = os.path.join(new_base_model_dir, f'fold_{fold}/data/')
filtered_data_path = os.path.join(df_dir, 'all_data_with_SI.csv')


protein_names = []
patch_indices = []
patch_summaries = []

with open(input_file_path, 'r') as file:
    lines = file.readlines()
    current_protein_name = None
    for line in lines:
        line = line.strip()
        if line.startswith('Patches for protein'):
            current_protein_name = line.split(' ')[3][:-1]
        elif line.startswith('indices:'):
            indices = line.split('[')[1].split(']')[0].split(', ')
            indices = [int(idx) for idx in indices]
            protein_names.append(current_protein_name)
            patch_indices.append(indices)

filtered_combined_df = pd.read_csv(filtered_data_path)
filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

for protein_name, indices in zip(protein_names, patch_indices):
    protein_data = filtered_combined_df[filtered_combined_df['name'] == protein_name]

    embeddings = []
    for idx in indices:
        residue_row = protein_data[protein_data['ResID'] == idx]
        if not residue_row.empty:
            embedding = residue_row.iloc[0, -321:-1].values
            embeddings.append(embedding)

    if len(embeddings) == 3:
        patch_embedding_sum = np.sum(embeddings, axis=0)
        patch_summary = {
            'Protein_Name': protein_name,
            'Patch_Indices': indices,
            'Status': 'positive'
        }

        for i, value in enumerate(patch_embedding_sum):
            patch_summary[f'Embedding_{i+1}'] = value

        patch_summaries.append(patch_summary)

summary_df = pd.DataFrame(patch_summaries)
print(summary_df)

In [ ]:
summary_df

In [ ]:
base_model_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_v4"
new_base_model_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_new_method/"
fold = 3
txt_files_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
filtered_data_path = os.path.join(new_base_model_dir, f'fold_{fold}/data/all_data_with_SI.csv')
output_file_path_neg = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/negative_testing_patch_index.csv')

filtered_combined_df = pd.read_csv(filtered_data_path)
filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

patch_summaries_neg = []

for i in range(1, 5):
    input_file_path = os.path.join(txt_files_dir, f'cnn_negative_patch_3points_part{i}.txt')

    with open(input_file_path, 'r') as file:
        lines = file.readlines()
        current_protein_name = None
        for line in lines:
            line = line.strip()
            if line.startswith('Patches for protein'):
                current_protein_name = line.split(' ')[3][:-1]
            elif line.startswith('indices:'):
                indices = line.split('[')[1].split(']')[0].split(', ')
                indices = [int(idx) for idx in indices]

                protein_data = filtered_combined_df[filtered_combined_df['name'] == current_protein_name]

                embeddings = []
                for idx in indices:
                    residue_row = protein_data[protein_data['ResID'] == idx]
                    if not residue_row.empty:
                        embedding = residue_row.iloc[0, -321:-1].values
                        embeddings.append(embedding)

                if len(embeddings) == len(indices):
                    patch_embedding_sum = np.sum(embeddings, axis=0)

                    patch_summary = {
                        'Protein_Name': current_protein_name,
                        'Patch_Indices': indices,
                        'Status': 'negative'
                    }

                    for j, value in enumerate(patch_embedding_sum):
                        patch_summary[f'Embedding_{j+1}'] = value

                    patch_summaries_neg.append(patch_summary)

summary_df_neg = pd.DataFrame(patch_summaries_neg)
print(summary_df_neg)

In [ ]:
combined_summary_df = pd.concat([summary_df, summary_df_neg], ignore_index=True)

In [ ]:
combined_summary_df

In [ ]:
output_file_combined = os.path.join(new_base_model_dir, f'fold_{fold}/data', 'combined_patch_summary.csv')
combined_summary_df.to_csv(output_file_combined, index=False)

In [ ]:
new_base_model_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_new_method/"
fold = 3
output_file_combined = os.path.join(new_base_model_dir, f'fold_{fold}/data', 'combined_patch_summary.csv')
combined_summary_df = pd.read_csv(output_file_combined)

In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import matplotlib.pyplot as plt

embedding_columns = [f'Embedding_{i}' for i in range(1, 321)]
X = combined_summary_df[embedding_columns].to_numpy()
y = combined_summary_df['Status'].replace({'positive': 1, 'negative': 0}).to_numpy()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

X_train, y_train = shuffle(X_train, y_train, random_state=42)

print(f"Training data shape: {X_train.shape}, Labels shape: {y_train.shape}")
print(f"Validation data shape: {X_val.shape}, Labels shape: {y_val.shape}")

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    verbose=1,
    callbacks=[early_stopping]
)

output_dir = './drive/MyDrive/@antigen_epitope/abdb_new_method/fold_3/data_processed_step1_with_SI/'
os.makedirs(output_dir, exist_ok=True)
model_save_path = os.path.join(output_dir, 'classification_model_with_val.h5')
model.save(model_save_path)
print(f"Model saved to {model_save_path}")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
y_pred_probs = model.predict(X_val).flatten()

fpr, tpr, thresholds = roc_curve(y_val, y_pred_probs)
J = tpr - fpr
ix = np.argmax(J)
best_threshold = thresholds[ix]

y_pred_labels = (y_pred_probs >= best_threshold).astype(int)

accuracy = accuracy_score(y_val, y_pred_labels)
precision = precision_score(y_val, y_pred_labels)
recall = recall_score(y_val, y_pred_labels)
f1 = f1_score(y_val, y_pred_labels)
auc = roc_auc_score(y_val, y_pred_probs)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Best Cutoff (Threshold): {best_threshold:.3f}")

plt.figure()
plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % auc)
plt.plot([0, 1], [0, 1], 'k--')
plt.scatter(fpr[ix], tpr[ix], marker='o', color='red', label='Best threshold')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()